# 09 — Phase 3 with FLORES-Derived Features

**Research question:** Does the capacity competition mechanism (Phase 3) also occur when we ablate *general-purpose* language features identified on FLORES, or only MGSM-specific features?

**Method:**
1. Run Phase 1 feature-ID on FLORES-200 activations (monolinguality + probe → A∩B)
2. Compare FLORES-derived vs MGSM-derived candidates (Jaccard overlap)
3. Run Phase 3 experiments using FLORES-derived features as the ablation set:
   - **(a) Capacity competition:** Do FLORES-feature ablations release dormant reasoning features?
   - **(c) Attention disruption:** Does attention entropy shift?
   - **(b) Circuit attribution:** What downstream features shift at L22/L29?
4. Compare to MGSM-derived Phase 3 results

**Interpretation:**
- If FLORES-derived ablation triggers the same capacity competition → mechanism is general
- If only MGSM-derived features trigger it → mechanism is domain-specific

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
    DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
else:
    DRIVE_RESULTS = None

if IN_COLAB:
    REPO_DIR = '/content/nlp-project'
    if not Path(REPO_DIR).exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
    os.chdir(REPO_DIR)
    !pip install -q 'numpy>=2.0' datasets networkx -e .
    Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
else:
    REPO_DIR = os.getcwd()
    from dotenv import load_dotenv
    load_dotenv()
    assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing in .env at repo root'

for _mod_name in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_mod_name]

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

from src.config import (
    TARGET_LANGUAGES, SAE_SUBSET_LAYERS, SAE_WIDTH_16K,
    RESULTS_DIR as _LOCAL_RESULTS_DIR, D_MODEL, MODEL_ID,
)
from src.data import load_mgsm
from src.model import load_saes_at_layers, get_decoder_layers
from src.extraction import extract_residual_activations, encode_activations_through_sae
from src.monolinguality import (
    compute_monolinguality, identify_language_features,
    train_language_probe, probe_language_features,
)
from src.intervention import directional_ablation, get_sae_decoder_directions

torch.manual_seed(0)
np.random.seed(0)

PRIMARY_LAYER = 17
DOWNSTREAM_LAYERS = [22, 29]
TOP_K_FEATURE_ID = 50
N_DEV = 50
N_ATTENTION = 30
N_CIRCUIT = 10

if DRIVE_RESULTS is not None:
    RESULTS_DIR = DRIVE_RESULTS
    print(f'Saving results to Drive: {RESULTS_DIR}')
else:
    RESULTS_DIR = _LOCAL_RESULTS_DIR
    print(f'Saving results locally: {RESULTS_DIR}')

FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True, parents=True)

## 1. Load model (with eager attention) and SAEs

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto',
    token=os.environ['HF_TOKEN'], attn_implementation='eager',
)
model.eval()
DECODER_LAYERS = get_decoder_layers(model)
print(f'Model loaded with eager attention. Layers: {len(DECODER_LAYERS)}')

saes = load_saes_at_layers(
    layers=SAE_SUBSET_LAYERS, width=SAE_WIDTH_16K, l0_target='medium',
)
sae17 = saes[17]

## 2. Load prior results and MGSM-derived Phase 3 for comparison

In [ ]:
phase1 = torch.load(RESULTS_DIR / 'phase1_features.pt', weights_only=False)
phase2b = torch.load(RESULTS_DIR / 'phase2_ablation.pt', weights_only=False)
phase3_mgsm = torch.load(RESULTS_DIR / 'phase3_interaction.pt', weights_only=False)

mgsm_intersection = phase1['intersection_features'][PRIMARY_LAYER]
reasoning_features_per_layer = phase1['reasoning_features']
best_k = phase2b['best_k']

REASONING_FEATS_L17 = reasoning_features_per_layer[17]
print(f'Reasoning features at L17: {len(REASONING_FEATS_L17)}')
print(f'Best k from Phase 2b: {best_k}')

# MGSM Phase 3 capacity summary for comparison
mgsm_capacity_summary = {}
for lang in TARGET_LANGUAGES:
    mgsm_capacity_summary[lang] = pd.DataFrame(phase3_mgsm['capacity']['summary'][lang])
    n_sig = int((mgsm_capacity_summary[lang]['p_value'] < 0.05).sum())
    print(f'  MGSM Phase 3 {lang}: {n_sig} sig features')

## 3. Extract FLORES features and run Phase 1

In [ ]:
from datasets import load_dataset

FLORES_LANG_MAP = {
    'en': 'eng_Latn',
    'zh': 'cmn_Hans',
    'es': 'spa_Latn',
    'bn': 'ben_Beng',
    'sw': 'swh_Latn',
}

def make_prompt(text):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': text}],
        tokenize=False,
        add_generation_prompt=True,
    )

flores_texts = {}
for lang, flores_code in FLORES_LANG_MAP.items():
    ds = load_dataset('openlanguagedata/flores_plus', flores_code, split='devtest')
    flores_texts[lang] = ds['text']
    print(f'  {lang} ({flores_code}): {len(flores_texts[lang])} sentences')

In [ ]:
flores_prompts = {
    lang: [make_prompt(s) for s in sents]
    for lang, sents in flores_texts.items()
}

# Extract residual activations at L17
flores_residuals = {}
for lang in TARGET_LANGUAGES:
    print(f'Extracting {lang} ({len(flores_prompts[lang])} sentences)...')
    acts = extract_residual_activations(
        model, tokenizer,
        flores_prompts[lang],
        layers=[PRIMARY_LAYER],
        positions='last',
    )
    flores_residuals[lang] = acts[PRIMARY_LAYER]
    print(f'  {lang}: {flores_residuals[lang].shape}')

# Encode through SAE
flores_feats = {}
for lang in TARGET_LANGUAGES:
    flores_feats[lang] = encode_activations_through_sae(
        flores_residuals[lang], sae17,
    )
    print(f'  {lang} SAE features: {flores_feats[lang].shape}')

del flores_residuals
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Method A: monolinguality on FLORES
mono_flores = compute_monolinguality(flores_feats)
method_a_flores = identify_language_features(mono_flores, top_k=TOP_K_FEATURE_ID)

# Method B: language probe on FLORES
print('Training language probe on FLORES SAE features...')
flores_probe, flores_importances = train_language_probe(flores_feats)
languages_sorted = sorted(TARGET_LANGUAGES)
method_b_flores = probe_language_features(
    flores_probe, flores_importances, languages_sorted, top_k=TOP_K_FEATURE_ID
)

# A \u2229 B intersection
flores_intersection = {}
print('\nFLORES A\u2229B intersection sizes:')
for lang in TARGET_LANGUAGES:
    set_a = set(method_a_flores[lang])
    set_b = set(method_b_flores[lang])
    isect = sorted(set_a & set_b)
    flores_intersection[lang] = isect
    print(f'  {lang}: {len(isect)} features')

In [ ]:
# Cross-domain overlap
print('Cross-domain feature overlap (FLORES A\u2229B vs MGSM A\u2229B):')
print(f'{"lang":>4}  {"MGSM":>5}  {"FLORES":>6}  {"overlap":>7}  {"Jaccard":>8}')
print('-' * 40)

overlap_results = {}
for lang in TARGET_LANGUAGES:
    mgsm_set = set(mgsm_intersection[lang])
    flores_set = set(flores_intersection[lang])
    overlap = mgsm_set & flores_set
    union = mgsm_set | flores_set
    jaccard = len(overlap) / len(union) if union else 0
    overlap_results[lang] = {
        'mgsm_count': len(mgsm_set),
        'flores_count': len(flores_set),
        'overlap_count': len(overlap),
        'jaccard': jaccard,
    }
    print(f'{lang:>4}  {len(mgsm_set):>5}  {len(flores_set):>6}  '
          f'{len(overlap):>7}  {jaccard:>8.3f}')

## 4. Build FLORES-derived ablation sets

Use the same selection logic as Phase 2b: A\u2229B intersection, backfilled with Method A, capped at k.

In [ ]:
def select_flores_features(lang, k=best_k):
    """Select up to k FLORES-derived features: A cap B first, then Method A backfill."""
    out = list(flores_intersection[lang])
    for f in method_a_flores[lang]:
        if f not in out:
            out.append(f)
    return out[:k]

flores_ablation_features = {l: select_flores_features(l) for l in TARGET_LANGUAGES}
print('FLORES-derived ablation feature sets per language:')
for l in TARGET_LANGUAGES:
    print(f'  {l}: {len(flores_ablation_features[l])} features \u2014 {flores_ablation_features[l]}')

In [ ]:
# Load MGSM for the forward passes
mgsm = load_mgsm(TARGET_LANGUAGES)
print(f'MGSM loaded: {sum(len(mgsm[l]) for l in TARGET_LANGUAGES)} problems')

## 5. Helpers — forward pass with optional ablation hook

Same helpers as notebook 05 (Phase 3), copied here for self-containedness.

In [ ]:
def make_ablation_hook(directions: torch.Tensor, input_length: int):
    """Hook: project out the span of `directions` from the last input token."""
    pos = input_length - 1
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            hs = output
            is_tuple = False
        else:
            hs = output[0]
            is_tuple = True
        if hs.dim() == 3 and hs.shape[1] > pos:
            hs[:, pos, :] = directional_ablation(hs[:, pos, :], directions)
        if is_tuple:
            return (hs,) + output[1:]
        return hs
    return hook


def forward_collect(prompt: str, layers_to_collect: list[int],
                    ablation_layer: int = None,
                    ablation_dirs: torch.Tensor = None,
                    want_attentions: bool = False):
    """Single forward pass. Returns residuals at specified layers (last-token)
    and optionally attention probabilities."""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    handles = []
    if ablation_layer is not None and ablation_dirs is not None and len(ablation_dirs) > 0:
        handles.append(DECODER_LAYERS[ablation_layer].register_forward_hook(
            make_ablation_hook(ablation_dirs.to(model.device), input_len)))
    try:
        with torch.no_grad():
            out = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=want_attentions,
                use_cache=False,
            )
    finally:
        for h in handles:
            h.remove()

    resid = {layer: out.hidden_states[layer + 1][:, -1, :].cpu().float()
             for layer in layers_to_collect}
    attns = None
    if want_attentions:
        attns = [a.detach().cpu().float() for a in out.attentions]
    return {'resid': resid, 'attns': attns, 'input_len': input_len}

## 6. Experiment (a) — Capacity competition with FLORES features

For each (lang, problem), forward twice (clean / ablated at L17 using FLORES-derived features), then SAE-encode the L17 residual and compare reasoning-feature activations.

In [ ]:
print(f'Tracking {len(REASONING_FEATS_L17)} reasoning features at L17')

clean_feats   = {lang: [] for lang in TARGET_LANGUAGES}
ablated_feats = {lang: [] for lang in TARGET_LANGUAGES}

for lang in TARGET_LANGUAGES:
    dirs = get_sae_decoder_directions(sae17, flores_ablation_features[lang]).to(model.device)
    print(f'\n=== {lang}: ablating {len(flores_ablation_features[lang])} FLORES features at L17 ===')
    for i in tqdm(range(N_DEV), desc=f'{lang}'):
        prompt = make_prompt(mgsm[lang][i]['question'])
        clean = forward_collect(prompt, [17], ablation_layer=None)
        ablt  = forward_collect(prompt, [17], ablation_layer=17, ablation_dirs=dirs)
        clean_f = encode_activations_through_sae(clean['resid'][17], sae17)[0]
        ablt_f  = encode_activations_through_sae(ablt['resid'][17],  sae17)[0]
        clean_feats[lang].append(clean_f[REASONING_FEATS_L17])
        ablated_feats[lang].append(ablt_f[REASONING_FEATS_L17])
    clean_feats[lang]   = torch.stack(clean_feats[lang])
    ablated_feats[lang] = torch.stack(ablated_feats[lang])
    print(f'  shapes: clean {tuple(clean_feats[lang].shape)}, ablated {tuple(ablated_feats[lang].shape)}')
    # Checkpoint
    torch.save({'clean': clean_feats, 'ablated': ablated_feats},
               RESULTS_DIR / 'phase3a_flores_capacity_partial.pt')

In [ ]:
# Statistical test per (lang, reasoning feature)
flores_capacity_summary = {}
for lang in TARGET_LANGUAGES:
    rows = []
    cf = clean_feats[lang].numpy()
    af = ablated_feats[lang].numpy()
    for j, feat_idx in enumerate(REASONING_FEATS_L17):
        diff = af[:, j] - cf[:, j]
        if np.allclose(diff, 0):
            p = 1.0; stat = 0.0
        else:
            try:
                stat, p = wilcoxon(af[:, j], cf[:, j])
            except ValueError:
                stat, p = 0.0, 1.0
        rows.append({
            'feature': int(feat_idx),
            'mean_clean': float(cf[:, j].mean()),
            'mean_ablated': float(af[:, j].mean()),
            'mean_delta': float(diff.mean()),
            'p_value': float(p),
        })
    df = pd.DataFrame(rows)
    flores_capacity_summary[lang] = df
    n_up   = int((df['mean_delta'] > 0).sum())
    n_down = int((df['mean_delta'] < 0).sum())
    n_sig  = int((df['p_value'] < 0.05).sum())
    print(f'  {lang}: {n_up} \u2191, {n_down} \u2193, {n_sig} significant (p<0.05) of {len(df)}')

In [ ]:
# Compare: does feature 96 (the headline MGSM result) also activate
# when ablating FLORES-derived features?
print('\nFeature 96 (headline dormant reasoning feature) comparison:')
print(f'{"lang":>4}  {"MGSM clean":>11}  {"MGSM ablated":>13}  '
      f'{"FLORES clean":>13}  {"FLORES ablated":>15}')
print('-' * 70)
for lang in TARGET_LANGUAGES:
    # MGSM Phase 3
    mgsm_df = mgsm_capacity_summary[lang]
    mgsm_row = mgsm_df[mgsm_df['feature'] == 96]
    if len(mgsm_row) > 0:
        mc = mgsm_row.iloc[0]['mean_clean']
        ma = mgsm_row.iloc[0]['mean_ablated']
    else:
        mc, ma = float('nan'), float('nan')
    # FLORES Phase 3
    flores_df = flores_capacity_summary[lang]
    flores_row = flores_df[flores_df['feature'] == 96]
    if len(flores_row) > 0:
        fc = flores_row.iloc[0]['mean_clean']
        fa = flores_row.iloc[0]['mean_ablated']
    else:
        fc, fa = float('nan'), float('nan')
    print(f'{lang:>4}  {mc:>11.1f}  {ma:>13.1f}  {fc:>13.1f}  {fa:>15.1f}')

## 7. Experiment (c) — Attention disruption with FLORES features

In [ ]:
ATTN_LAYERS = SAE_SUBSET_LAYERS

def attention_entropy_last_query(attn_layer: torch.Tensor) -> torch.Tensor:
    """Given attention probs (1, H, T, T), return per-head entropy at q=T-1."""
    p = attn_layer[0, :, -1, :].clamp_min(1e-12)
    return -(p * p.log()).sum(dim=-1)

entropy_clean   = {layer: {l: [] for l in TARGET_LANGUAGES} for layer in ATTN_LAYERS}
entropy_ablated = {layer: {l: [] for l in TARGET_LANGUAGES} for layer in ATTN_LAYERS}

for lang in TARGET_LANGUAGES:
    dirs = get_sae_decoder_directions(sae17, flores_ablation_features[lang]).to(model.device)
    for i in tqdm(range(N_ATTENTION), desc=f'attn {lang}'):
        prompt = make_prompt(mgsm[lang][i]['question'])
        clean = forward_collect(prompt, [], ablation_layer=None, want_attentions=True)
        ablt  = forward_collect(prompt, [], ablation_layer=17,
                                ablation_dirs=dirs, want_attentions=True)
        for layer in ATTN_LAYERS:
            entropy_clean[layer][lang].append(attention_entropy_last_query(clean['attns'][layer]))
            entropy_ablated[layer][lang].append(attention_entropy_last_query(ablt['attns'][layer]))
    for layer in ATTN_LAYERS:
        entropy_clean[layer][lang]   = torch.stack(entropy_clean[layer][lang])
        entropy_ablated[layer][lang] = torch.stack(entropy_ablated[layer][lang])
    torch.save({'clean': entropy_clean, 'ablated': entropy_ablated},
               RESULTS_DIR / 'phase3c_flores_attention_partial.pt')

In [ ]:
# Aggregate: mean entropy delta per (layer, language)
flores_attn_summary = []
for layer in ATTN_LAYERS:
    for lang in TARGET_LANGUAGES:
        c = entropy_clean[layer][lang]
        a = entropy_ablated[layer][lang]
        delta = (a - c).mean().item()
        try:
            stat, p = wilcoxon(a.flatten().numpy(), c.flatten().numpy())
        except ValueError:
            stat, p = 0.0, 1.0
        flores_attn_summary.append({
            'layer': layer, 'lang': lang, 'mean_delta_entropy': delta,
            'p_value': float(p),
        })
flores_attn_df = pd.DataFrame(flores_attn_summary)
print('FLORES-derived attention entropy delta:')
print(flores_attn_df.pivot(index='layer', columns='lang', values='mean_delta_entropy').round(4))

## 8. Experiment (b) — Circuit attribution with FLORES features

In [ ]:
TOP_EDGE = 30
circuit_results = {lang: [] for lang in TARGET_LANGUAGES}

for lang in TARGET_LANGUAGES:
    dirs = get_sae_decoder_directions(sae17, flores_ablation_features[lang]).to(model.device)
    print(f'\n=== circuit attribution: {lang} ===')
    for i in tqdm(range(N_CIRCUIT), desc=f'circuit {lang}'):
        prompt = make_prompt(mgsm[lang][i]['question'])
        clean = forward_collect(prompt, DOWNSTREAM_LAYERS, ablation_layer=None)
        ablt  = forward_collect(prompt, DOWNSTREAM_LAYERS, ablation_layer=17, ablation_dirs=dirs)

        per_layer_edges = {}
        for layer in DOWNSTREAM_LAYERS:
            sae_d = saes[layer]
            f_clean = encode_activations_through_sae(clean['resid'][layer], sae_d)[0]
            f_ablt  = encode_activations_through_sae(ablt['resid'][layer],  sae_d)[0]
            delta = (f_ablt - f_clean).abs()
            top = torch.topk(delta, k=TOP_EDGE)
            per_layer_edges[layer] = [
                {'feature': int(top.indices[j]),
                 'delta': float(f_ablt[top.indices[j]] - f_clean[top.indices[j]]),
                 'abs_delta': float(top.values[j])}
                for j in range(TOP_EDGE)
            ]
        circuit_results[lang].append({
            'problem_idx': i, 'edges': per_layer_edges,
            'ablated_features_l17': flores_ablation_features[lang],
        })
    torch.save(circuit_results, RESULTS_DIR / 'phase3b_flores_circuit_partial.pt')

In [ ]:
from collections import defaultdict

edge_strength = defaultdict(list)
for lang in TARGET_LANGUAGES:
    for record in circuit_results[lang]:
        for downstream_layer, edges in record['edges'].items():
            for e in edges:
                key = (lang, downstream_layer, e['feature'])
                edge_strength[key].append(e['abs_delta'])

summary_rows = []
for (lang, layer, feat), vs in edge_strength.items():
    summary_rows.append({
        'lang': lang, 'downstream_layer': layer, 'downstream_feature': feat,
        'mean_abs_delta': float(np.mean(vs)), 'n_problems': len(vs),
    })
flores_edge_df = pd.DataFrame(summary_rows).sort_values('mean_abs_delta', ascending=False)
print('Top 20 strongest downstream effects (FLORES-derived ablation):')
print(flores_edge_df.head(20).to_string(index=False))

## 9. Comparison: FLORES-derived vs MGSM-derived Phase 3

In [ ]:
sns.set_theme(style='whitegrid', context='paper', font_scale=1.1)

# Figure 1: Capacity competition violins — MGSM vs FLORES side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (label, summary) in zip(axes, [
    ('MGSM-derived', mgsm_capacity_summary),
    ('FLORES-derived', flores_capacity_summary),
]):
    rows = []
    for lang in TARGET_LANGUAGES:
        for d in summary[lang]['mean_delta']:
            rows.append({'lang': lang, 'mean_delta': d})
    sns.violinplot(data=pd.DataFrame(rows), x='lang', y='mean_delta', ax=ax,
                   inner='quartile', density_norm='width')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'{label} ablation')
    ax.set_ylabel('\u0394 activation (ablated \u2212 clean)' if ax == axes[0] else '')
    ax.set_xlabel('Language')

fig.suptitle('Capacity competition: reasoning feature activation shift at L17', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_flores_vs_mgsm_capacity.png', dpi=150)
plt.show()

In [ ]:
# Figure 2: Attention entropy heatmaps side by side
mgsm_attn_df = pd.DataFrame(phase3_mgsm['attention']['summary'])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, (label, adf) in zip(axes, [
    ('MGSM-derived', mgsm_attn_df),
    ('FLORES-derived', flores_attn_df),
]):
    heat = adf.pivot(index='layer', columns='lang', values='mean_delta_entropy')
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
                cbar_kws={'label': '\u0394 entropy'}, ax=ax)
    ax.set_title(f'{label} ablation')

fig.suptitle('Attention entropy shift (ablated \u2212 clean)', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_flores_vs_mgsm_attention.png', dpi=150)
plt.show()

In [ ]:
# Summary table
print('\n' + '=' * 70)
print('SUMMARY: FLORES-derived vs MGSM-derived Phase 3')
print('=' * 70)

print('\n(a) Capacity competition \u2014 significant reasoning features per language:')
print(f'{"lang":>4}  {"MGSM sig":>9}  {"FLORES sig":>11}')
for lang in TARGET_LANGUAGES:
    m_sig = int((mgsm_capacity_summary[lang]['p_value'] < 0.05).sum())
    f_sig = int((flores_capacity_summary[lang]['p_value'] < 0.05).sum())
    print(f'{lang:>4}  {m_sig:>9}  {f_sig:>11}')

print('\n(c) Attention entropy \u2014 mean \u0394 at L22 (peak intervention layer):')
for lang in TARGET_LANGUAGES:
    m_delta = mgsm_attn_df[(mgsm_attn_df['layer']==22) & (mgsm_attn_df['lang']==lang)]['mean_delta_entropy'].values[0]
    f_delta = flores_attn_df[(flores_attn_df['layer']==22) & (flores_attn_df['lang']==lang)]['mean_delta_entropy'].values[0]
    print(f'  {lang}: MGSM {m_delta:+.4f}, FLORES {f_delta:+.4f}')

print('\nInterpretation:')
# Check if FLORES triggers similar capacity competition
mgsm_total_sig = sum((mgsm_capacity_summary[l]['p_value'] < 0.05).sum() for l in TARGET_LANGUAGES)
flores_total_sig = sum((flores_capacity_summary[l]['p_value'] < 0.05).sum() for l in TARGET_LANGUAGES)
ratio = flores_total_sig / max(mgsm_total_sig, 1)
if ratio > 0.7:
    print(f'FLORES-derived ablation triggers comparable capacity competition ({ratio:.0%} of MGSM).')
    print('-> The mechanism is GENERAL, not domain-specific.')
elif ratio > 0.3:
    print(f'FLORES-derived ablation triggers partial capacity competition ({ratio:.0%} of MGSM).')
    print('-> The mechanism is partially general, partially domain-specific.')
else:
    print(f'FLORES-derived ablation triggers minimal capacity competition ({ratio:.0%} of MGSM).')
    print('-> The mechanism is domain-specific.')

## 10. Save results

In [ ]:
payload = {
    'config': {
        'primary_layer': PRIMARY_LAYER,
        'downstream_layers': DOWNSTREAM_LAYERS,
        'attention_layers': list(ATTN_LAYERS),
        'n_dev': N_DEV,
        'n_attention': N_ATTENTION,
        'n_circuit': N_CIRCUIT,
        'best_k': best_k,
        'reasoning_features_l17': REASONING_FEATS_L17,
        'flores_lang_map': FLORES_LANG_MAP,
        'top_k_feature_id': TOP_K_FEATURE_ID,
    },
    'flores_intersection': flores_intersection,
    'flores_ablation_features': flores_ablation_features,
    'overlap_results': overlap_results,
    'capacity': {
        'clean_feats':   {l: t for l, t in clean_feats.items()},
        'ablated_feats': {l: t for l, t in ablated_feats.items()},
        'summary': {l: flores_capacity_summary[l].to_dict('records') for l in TARGET_LANGUAGES},
    },
    'attention': {
        'entropy_clean':   {layer: dict(entropy_clean[layer]) for layer in ATTN_LAYERS},
        'entropy_ablated': {layer: dict(entropy_ablated[layer]) for layer in ATTN_LAYERS},
        'summary': flores_attn_df.to_dict('records'),
    },
    'circuit': {
        'per_problem': circuit_results,
        'edges': flores_edge_df.to_dict('records'),
    },
}

out = RESULTS_DIR / 'phase3_flores_interaction.pt'
torch.save(payload, out)
print(f'Saved to {out}')
if DRIVE_RESULTS is not None:
    torch.save(payload, DRIVE_RESULTS / 'phase3_flores_interaction.pt')
print(f'Figures saved to {FIG_DIR}/')